# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# ============================================================================
# SETUP
# ============================================================================

import os
import sys
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

# Add exercise folder to path for module imports
sys.path.insert(0, 'exercise')

from utils.prompts import create_system_prompt, EXPERTISE_LEVELS
from utils.model_registry import TechnicalAssistant
from utils.tools import explain_error, suggest_improvements, TOOLS, TOOL_REGISTRY, handle_tool_calls

load_dotenv(override=True)

assistant = TechnicalAssistant()
client = OpenAI()

print("Setup complete")
print(f"  Models: {assistant.get_available_models()}")
print(f"  Tools: {list(TOOL_REGISTRY.keys())}")
print(f"  Expertise levels: {EXPERTISE_LEVELS}")


Ollama detected and available
Setup complete
  Models: ['GPT', 'Ollama']
  Tools: ['explain_error', 'suggest_improvements']
  Expertise levels: {1: 'beginner', 2: 'intermediate', 3: 'advanced'}


---

## Phase 1: Basic Gradio UI

Multi-model support, streaming, expertise-based system prompts.


In [2]:
# ============================================================================
# PHASE 1: BASIC GRADIO UI WITH STREAMING
# ============================================================================
# Multi-model support, streaming, expertise-based prompts

def chat_phase1(message, history, model, expertise_slider):
    """
    Basic chat callback with streaming.
    Uses streaming-only mode (no tool calling).
    """
    expertise = EXPERTISE_LEVELS.get(int(expertise_slider), "intermediate")
    yield from assistant.chat_streaming_only(message, history, model, expertise)

# Build Phase 1 interface
with gr.Blocks(title="Technical Assistant - Phase 1", theme=gr.themes.Soft()) as demo_phase1:
    gr.Markdown("# Technical Question Assistant (Phase 1)")
    gr.Markdown("Multi-model chat with expertise-adaptive responses.")
    
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=assistant.get_available_models(),
            value="GPT",
            label="Model"
        )
        expertise_slider = gr.Slider(
            minimum=1, maximum=3, step=1, value=2,
            label="Expertise Level",
            info="1=Beginner, 2=Intermediate, 3=Advanced"
        )
    
    gr.ChatInterface(
        fn=chat_phase1,
        type="messages",
        additional_inputs=[model_dropdown, expertise_slider],
        examples=[
            ["Explain what a Python decorator does"],
            ["What's the difference between a list and a tuple?"],
            ["How does async/await work in Python?"]
        ]
    )

print("Phase 1 UI ready - uncomment demo_phase1.launch() to run")
#demo_phase1.launch()


Phase 1 UI ready - uncomment demo_phase1.launch() to run


---

## Phase 2: Tool Calling

`explain_error` and `suggest_improvements` tools. LLM invokes them automatically.


In [3]:
# ============================================================================
# PHASE 2: TOOL CALLING
# ============================================================================
# LLM automatically invokes tools when appropriate

# Test tool directly
print("=" * 60)
print("Testing explain_error tool:")
print("=" * 60)
print(explain_error("TypeError: 'NoneType' object is not subscriptable"))

# Test via LLM (should trigger tool calling)
print("\n" + "=" * 60)
print("Testing chat with tool calling:")
print("=" * 60)

test_message = "Can you explain this Python error? IndexError: list index out of range"
print(f"Question: {test_message}\n")

for chunk in assistant.chat(test_message, [], "GPT", "intermediate"):
    final_response = chunk

print(f"Response:\n{final_response[:500]}...")

print("\nTool calling verified")


Testing explain_error tool:
**Error Type:** TypeError
**Message:** 'NoneType' object is not subscriptable

**What this means:**
You tried to perform an operation on a value of the wrong type.

**Common causes:**
- Adding/concatenating incompatible types (e.g., string + integer)
- Calling a non-callable object like a function
- Passing wrong number of arguments to a function
- Using None where a value was expected

**How to fix it:**
- Check the types of your variables using type()
- Convert values to compatible types (str(), int(), float())
- Check if a variable might be None before using it

Testing chat with tool calling:
Question: Can you explain this Python error? IndexError: list index out of range

Response:
The **IndexError: list index out of range** is a common error in Python that indicates you're trying to access an index in a list that doesn’t exist. Let's break down what this means, why it happens, and how you can avoid it.

### Understanding the Error

1. **Zero-Based Inde

---

## Phase 3: Audio I/O

Speech-to-text (Whisper) and text-to-speech (OpenAI TTS).


In [4]:
# ============================================================================
# PHASE 3: AUDIO I/O
# ============================================================================
# Speech-to-text (Whisper) and text-to-speech (OpenAI TTS)

def transcribe_audio(audio_input) -> str:
    """
    Transcribe audio to text using OpenAI Whisper.
    
    Handles both filepath strings and numpy tuple (sample_rate, audio_data).
    """
    if audio_input is None:
        return ""
    
    import tempfile
    import os
    import numpy as np
    
    try:
        # Handle tuple (sample_rate, audio_data) from Gradio
        if isinstance(audio_input, tuple):
            import wave
            
            sample_rate, audio_data = audio_input
            
            # Validate sample rate
            if sample_rate is None or sample_rate < 8000:
                return f"Error: Invalid sample rate ({sample_rate}). Please try again."
            
            # Validate audio data
            if audio_data is None:
                return "Error: No audio data received. Please record again."
            
            audio_data = np.array(audio_data, dtype=np.float32)
            
            if len(audio_data) == 0:
                return "Error: Empty audio data. Please record again."
            
            # Check audio duration (should be at least 0.5 seconds)
            duration = len(audio_data) / sample_rate
            if duration < 0.5:
                return f"Error: Audio too short ({duration:.2f}s). Please record at least 1 second."
            
            # Resample if needed (Whisper works best with 16kHz, but accepts 8-48kHz)
            # Only resample if way off (scipy might not be available)
            if sample_rate < 8000 or sample_rate > 48000:
                try:
                    from scipy import signal
                    target_rate = 16000
                    num_samples = int(len(audio_data) * target_rate / sample_rate)
                    audio_data = signal.resample(audio_data, num_samples)
                    sample_rate = target_rate
                except ImportError:
                    # scipy not available, use original sample rate
                    pass
            
            # Convert to 1D array if needed
            if len(audio_data.shape) > 1:
                audio_data = audio_data.flatten()
            
            # Check if audio is mostly silence
            max_amplitude = np.abs(audio_data).max()
            if max_amplitude < 0.01:  # Very quiet
                return "Error: Audio appears to be silence. Please speak louder or check your microphone."
            
            # Normalize to [-1, 1] range if needed, then convert to int16
            if audio_data.dtype == np.float32 or audio_data.dtype == np.float64:
                # Clamp to [-1, 1] range
                audio_data = np.clip(audio_data, -1.0, 1.0)
                # Convert to int16
                audio_data = (audio_data * 32767).astype(np.int16)
            elif audio_data.dtype != np.int16:
                # Convert other integer types to int16
                if audio_data.dtype in [np.int32, np.int64]:
                    # Scale down if needed
                    max_val = np.abs(audio_data).max()
                    if max_val > 32767:
                        audio_data = (audio_data / (max_val / 32767)).astype(np.int16)
                    else:
                        audio_data = audio_data.astype(np.int16)
                else:
                    audio_data = audio_data.astype(np.int16)
            
            # Save to temp WAV file
            with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp_file:
                with wave.open(tmp_file.name, 'wb') as wav_file:
                    wav_file.setnchannels(1)  # Mono
                    wav_file.setsampwidth(2)  # 16-bit
                    wav_file.setframerate(int(sample_rate))
                    wav_file.writeframes(audio_data.tobytes())
                audio_path = tmp_file.name
        else:
            # It's a filepath string
            audio_path = audio_input
            if not os.path.exists(audio_path):
                return "Error: Audio file not found. Please record again."
            if os.path.getsize(audio_path) == 0:
                return "Error: Audio file is empty. Please record again."
        
        # Transcribe with Whisper
        with open(audio_path, "rb") as f:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=f,
                language="en"
            )
        
        result = transcript.text.strip()
        
        # Validate transcription result
        if not result or len(result) < 2:
            return "Error: Transcription returned empty or too short. Please try recording again with clearer audio."
        
        return result
        
    except Exception as e:
        import traceback
        error_msg = f"Error transcribing: {str(e)}"
        # Only show full traceback in debug mode
        return error_msg


def speak_response(text: str) -> str:
    """Convert text to speech using OpenAI TTS."""
    if not text:
        return None
    try:
        # Truncate to avoid TTS timeouts and costs
        if len(text) > 1000:
            text = text[:1000] + "... (truncated)"
        
        response = client.audio.speech.create(
            model="tts-1",
            voice="nova",
            input=text
        )
        
        # Store TTS output in exercise/outputs/ (add *.mp3 to .gitignore)
        import os
        output_dir = os.path.join("exercise", "outputs")
        os.makedirs(output_dir, exist_ok=True)
        audio_path = os.path.join(output_dir, "response.mp3")
        
        response.stream_to_file(audio_path)
        return audio_path
    except Exception as e:
        print(f"TTS Error: {str(e)}")
        return None


print("Audio functions ready")


Audio functions ready


---

## Final: Complete Application

Everything combined: multi-model, streaming, expertise prompts, tool calling, audio I/O.



In [ ]:
# ============================================================================
# FINAL: COMPLETE APPLICATION
# ============================================================================
# Multi-model, streaming, expertise prompts, tool calling, audio I/O

last_response = ""

def chat_complete(message, history, model, expertise_slider):
    """Chat with streaming, tools, and response tracking for TTS."""
    global last_response
    expertise = EXPERTISE_LEVELS.get(int(expertise_slider), "intermediate")
    
    response = ""
    for chunk in assistant.chat(message, history, model, expertise):
        response = chunk
        yield response
    
    last_response = response


def speak_last():
    """Convert last response to speech."""
    global last_response
    if last_response:
        return speak_response(last_response)
    return None


def transcribe_file(audio_path):
    """Transcribe audio file directly using filepath."""
    if audio_path is None:
        return "No audio provided. Please record or upload first."
    
    import os
    import wave
    import numpy as np
    
    if not os.path.exists(audio_path):
        return f"Error: File not found at {audio_path}"
    
    file_size = os.path.getsize(audio_path)
    if file_size == 0:
        return "Error: Audio file is empty. Please try again."
    
    # Analyze the audio file
    try:
        with wave.open(audio_path, 'rb') as wav:
            channels = wav.getnchannels()
            sample_width = wav.getsampwidth()
            framerate = wav.getframerate()
            n_frames = wav.getnframes()
            duration = n_frames / framerate
            
            # Read audio data to check amplitude
            raw_data = wav.readframes(n_frames)
            if sample_width == 2:
                audio_data = np.frombuffer(raw_data, dtype=np.int16)
            elif sample_width == 4:
                audio_data = np.frombuffer(raw_data, dtype=np.int32)
            else:
                audio_data = np.frombuffer(raw_data, dtype=np.uint8)
            
            max_amp = np.abs(audio_data).max()
            mean_amp = np.abs(audio_data).mean()
            
        print(f"[DEBUG] WAV info: {channels}ch, {sample_width*8}bit, {framerate}Hz, {duration:.2f}s")
        print(f"[DEBUG] Audio levels: max={max_amp}, mean={mean_amp:.1f}")
        
        if max_amp < 100:
            return f"Error: Audio appears to be silent (max amplitude: {max_amp}). Check your microphone settings."
            
    except Exception as e:
        print(f"[DEBUG] Could not analyze WAV: {e}")
    
    print(f"[DEBUG] Transcribing file: {audio_path} ({file_size} bytes)")
    
    try:
        with open(audio_path, "rb") as f:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=f
            )
        result = transcript.text.strip()
        print(f"[DEBUG] Whisper result: '{result}'")
        if not result:
            return "Transcription returned empty. Please try again with clearer audio."
        return result
    except Exception as e:
        return f"Error: {str(e)}"


# Custom CSS for ChatInterface-style example buttons
custom_css = """
.example-btn {
    border: 1px solid #374151 !important;
    background: transparent !important;
    color: #9ca3af !important;
    font-size: 0.85rem !important;
    padding: 8px 12px !important;
    border-radius: 8px !important;
    transition: all 0.2s ease !important;
    text-align: left !important;
    white-space: normal !important;
    height: auto !important;
    min-height: 50px !important;
}
.example-btn:hover {
    border-color: #6366f1 !important;
    background: rgba(99, 102, 241, 0.1) !important;
    color: #e5e7eb !important;
}
"""

# Gradio Blocks UI
with gr.Blocks(title="Technical Assistant", theme=gr.themes.Soft(), css=custom_css) as demo_final:
    gr.Markdown("# Technical Question Assistant")
    gr.Markdown("Multi-model chat with tool calling and voice I/O.")
    
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=assistant.get_available_models(),
            value="GPT",
            label="Model",
            scale=1
        )
        expertise_slider = gr.Slider(
            minimum=1, maximum=3, step=1, value=2,
            label="Expertise Level",
            info="1=Beginner (detailed), 2=Intermediate, 3=Advanced (concise)",
            scale=2
        )
    
    # State to capture audio path immediately when recorded/uploaded
    # This fixes the bug where Gradio loses audio component values during generator yields
    audio_path_state = gr.State(None)
    
    with gr.Accordion("Voice Input", open=False):
        gr.Markdown("Record or upload audio. Transcription will be sent directly to the chat.")
        
        with gr.Tab("Microphone"):
            mic_input = gr.Audio(
                sources=["microphone"],
                type="filepath",
                label="Record (click mic, speak, click stop)"
            )
        
        with gr.Tab("Upload File"):
            file_input = gr.Audio(
                sources=["upload"],
                type="filepath",
                label="Upload audio file"
            )
        
        send_to_chat_btn = gr.Button("Transcribe & Send to Chat", variant="primary")
        voice_status = gr.Textbox(
            label="Status",
            placeholder="",
            interactive=False,
            lines=1
        )
        
        # Capture audio path to state immediately when it changes
        def capture_audio(audio):
            return audio
        
        mic_input.change(fn=capture_audio, inputs=[mic_input], outputs=[audio_path_state])
        file_input.change(fn=capture_audio, inputs=[file_input], outputs=[audio_path_state])
    
    # Example prompts displayed above chat (clickable, styled like ChatInterface)
    gr.Markdown("**Examples** *(click to use)*")
    example_prompts = [
        "Explain what a Python decorator does",
        "What's the difference between a list and a tuple?",
        "How does async/await work in Python?",
        "Explain this error: TypeError: 'NoneType' object is not subscriptable",
    ]
    with gr.Row(equal_height=True):
        example_btns = [
            gr.Button(prompt, size="sm", elem_classes=["example-btn"]) 
            for prompt in example_prompts
        ]
    
    # Custom chat components (replaces ChatInterface for programmatic control)
    chatbot = gr.Chatbot(
        label="Chatbot",
        type="messages",
        height=400
    )
    
    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Type a message...",
            show_label=False,
            scale=9,
            container=False
        )
        submit_btn = gr.Button("Send", variant="primary", scale=1)
    
    # Chat state
    chat_history = gr.State([])
    
    def respond(message, history, model, expertise):
        """Handle user message and stream response."""
        if not message.strip():
            yield history, ""
            return
        
        # Add user message to history
        history = history + [{"role": "user", "content": message}]
        yield history, ""
        
        # Stream assistant response
        assistant_msg = ""
        for chunk in chat_complete(message, history[:-1], model, expertise):
            assistant_msg = chunk
        
        # Store for TTS
        global last_response
        last_response = assistant_msg
        
        history = history + [{"role": "assistant", "content": assistant_msg}]
        yield history, ""
    
    def use_example(prompt):
        """Fill input with example prompt."""
        return prompt
    
    # Wire up example buttons to fill input
    for btn, prompt in zip(example_btns, example_prompts):
        btn.click(fn=lambda p=prompt: p, inputs=[], outputs=[user_input])
    
    # Submit on button click or Enter
    submit_btn.click(
        fn=respond,
        inputs=[user_input, chat_history, model_dropdown, expertise_slider],
        outputs=[chatbot, user_input]
    ).then(lambda h: h, inputs=[chatbot], outputs=[chat_history])
    
    user_input.submit(
        fn=respond,
        inputs=[user_input, chat_history, model_dropdown, expertise_slider],
        outputs=[chatbot, user_input]
    ).then(lambda h: h, inputs=[chatbot], outputs=[chat_history])
    
    def transcribe_and_send(audio_path, history, model, expertise):
        """Transcribe audio and send directly to chat.
        
        Uses audio_path from gr.State which is captured immediately when audio is
        recorded/uploaded, avoiding the Gradio bug where component values are lost
        during generator yields.
        """
        if audio_path is None:
            yield history, "No audio provided. Record or upload first."
            return
        
        transcribed = transcribe_file(audio_path)
        
        if transcribed.startswith("Error") or not transcribed.strip():
            yield history, transcribed
            return
        
        # Show what was transcribed, then send to chat
        yield history, f"Transcribed: {transcribed}"
        
        for result in respond(transcribed, history, model, expertise):
            yield result[0], f"Sent: {transcribed}"
    
    # Wire up the send-to-chat button (uses audio_path_state instead of component values)
    send_to_chat_btn.click(
        fn=transcribe_and_send,
        inputs=[audio_path_state, chat_history, model_dropdown, expertise_slider],
        outputs=[chatbot, voice_status],
        show_progress="full"
    ).then(lambda h: h, inputs=[chatbot], outputs=[chat_history])
    
    with gr.Accordion("Voice Output", open=False):
        with gr.Row():
            speak_btn = gr.Button("Speak Last Response", variant="primary")
            audio_output = gr.Audio(
                label="Response Audio",
                type="filepath",
                autoplay=True
            )
        speak_btn.click(
            fn=speak_last,
            inputs=[],
            outputs=[audio_output]
        )
    
    gr.Markdown("---")
    gr.Markdown("*Week 2 Exercise | Built with Gradio and OpenAI*")

print("Complete application ready")
print("Uncomment demo_final.launch() below to run")

demo_final.launch(share=True)

# To deploy as standalone script:
# 1. Copy this cell to a .py file
# 2. Add imports from the setup cell
# 3. Run: python your_app.py


Complete application ready
Uncomment demo_final.launch() below to run
* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://39399db5d8b4132ac1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[DEBUG] WAV info: 1ch, 16bit, 44100Hz, 2.58s
[DEBUG] Audio levels: max=32767, mean=905.2
[DEBUG] Transcribing file: C:\Users\user\AppData\Local\Temp\gradio\bcf4a0656316164e6764a7042e19eab85d0973e0424f5c131ff7fe1bf2e4c866\audio.wav (227598 bytes)
[DEBUG] Whisper result: 'Olá, como estás?'


C:\Users\user\AppData\Local\Temp\ipykernel_21668\1375667639.py:144: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(audio_path)
